# 🤖 Supervised Learning — Retweet Prediction

End-to-end pipeline for predicting retweet counts using text embeddings and ensemble supervised learning.

**Pipeline overview:**
1. Load & inspect data
2. Clean text
3. Text embedding (sentence-transformers)
4. Train/test split
5. Random Forest — predict retweets, report train & test accuracy
6. Gradient Boosting — predict retweets, report train & test accuracy
7. Ensemble (soft voting average) — combined prediction

## 📂 1. Load Data

In [3]:
import sqlite3
import pandas as pd
from datetime import datetime
from tabulate import tabulate

DB_PATH = "../data/tweetsCleanedDB.sqlite"
TABLE_NAME = "tweets"

conn = sqlite3.connect(DB_PATH)

print(f"Processing table: {TABLE_NAME}")

cols = pd.read_sql_query(f"PRAGMA table_info({TABLE_NAME});", conn)["name"].tolist()

df = pd.read_sql(f"SELECT * FROM {TABLE_NAME}", conn)
print(f"Loaded {len(df)} rows")

if 'Timestamp' in df.columns:
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df['date'] = df['Timestamp'].dt.date
    df['time'] = df['Timestamp'].dt.time
    df['day_of_week'] = df['Timestamp'].dt.day_name()
    df['hour'] = df['Timestamp'].dt.hour
    df['year'] = df['Timestamp'].dt.year

print(f"\n{'='*50}")
print(f"Columns: {df.columns.tolist()}")
summary = pd.DataFrame([(TABLE_NAME, len(df))], columns=["table_name", "num_rows"])
print(tabulate(summary, headers="keys", tablefmt="github", showindex=False))

# Column datatypes
print(f"\n{'='*50}")
print("Column Datatypes:")
dtype_df = pd.DataFrame({
    "column": df.dtypes.index,
    "pandas_dtype": df.dtypes.values.astype(str),
    "sqlite_dtype": pd.read_sql_query(f"PRAGMA table_info({TABLE_NAME});", conn)["type"].tolist() + ["derived"] * (len(df.columns) - len(cols))
})
print(tabulate(dtype_df, headers="keys", tablefmt="github", showindex=False))

Processing table: tweets
Loaded 9976 rows

Columns: ['tweet_id', 'username', 'text', 'retweets', 'likes', 'timestamp', 'date', 'time', 'hour', 'day_name', 'year_week', 'year_month', 'year']
| table_name   |   num_rows |
|--------------|------------|
| tweets       |       9976 |

Column Datatypes:
| column     | pandas_dtype   | sqlite_dtype   |
|------------|----------------|----------------|
| tweet_id   | object         | TEXT           |
| username   | object         | TEXT           |
| text       | object         | TEXT           |
| retweets   | int64          | INTEGER        |
| likes      | int64          | INTEGER        |
| timestamp  | object         | TIMESTAMP      |
| date       | object         | TEXT           |
| time       | object         | TEXT           |
| hour       | int64          | INTEGER        |
| day_name   | object         | TEXT           |
| year_week  | object         | TEXT           |
| year_month | object         | TEXT           |
| year       | 

## 📦 2. Imports & Setup

In [4]:
import os
import re
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print('✅ All imports successful.')

✅ All imports successful.


## 🧹 3. Text Cleaning

Before computing embeddings, the raw text is cleaned to remove noise.

The following transformations are applied:

- **Lowercase** — normalises casing so "Twitter" and "twitter" are treated identically
- **URLs removed** — `http://...` and `www....` links carry no semantic meaning
- **Mentions removed** — `@username` tags are user-specific and not topically informative
- **Hashtags normalised** — `#` symbol stripped but the word is kept (e.g. `#AI` → `AI`)
- **RT prefix removed** — retweet markers (`RT`) are artefacts of the platform, not content
- **Special characters removed** — punctuation and symbols are stripped
- **Whitespace collapsed** — multiple spaces and newlines are reduced to a single space

> ⚠️ **Note:** The character filter `[^a-zA-Z0-9\s]` removes non-Latin characters. If your dataset contains non-English text (Arabic, Chinese, etc.), change the filter to `[^\w\s]` to preserve unicode word characters.

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)            # remove URLs
    text = re.sub(r'@\w+', '', text)                       # remove mentions
    text = re.sub(r'#(\w+)', r'\1', text)                  # strip # but keep word
    text = re.sub(r'^rt\s+', '', text, flags=re.MULTILINE) # remove RT prefix
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)            # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()               # collapse whitespace
    return text

df['text_clean'] = df['text'].apply(clean_text)
df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)

docs = df['text_clean'].tolist()
print(f'✅ Cleaned {len(docs)} documents.')
print(f'Sample: {docs[0]}')

✅ Cleaned 9976 documents.
Sample: party least receive say or single prevent prevent husband affect may himself cup style evening protect effect another themselves stage perform possible try tax share style television with successful much sell development economy effect


## 🔡 4. Text Embedding

Each cleaned tweet is encoded into a dense vector using a multilingual sentence-transformer model.
Embeddings are L2-normalised and cached to disk to avoid recomputing on subsequent runs.

- **Model:** `paraphrase-MiniLM-L3-v2` — 384-dimensional embeddings, fast, multilingual
- **Device:** GPU (CUDA/MPS) if available, otherwise CPU
- **Cache:** saved to `embeddings_cache.npy`

In [6]:
EMBED_MODEL = 'sentence-transformers/paraphrase-MiniLM-L3-v2'
CACHE_PATH  = 'embeddings_cache.npy'

# ── Auto-detect best available device ────────────────────────────────────────
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'▶ Using device: {device}')

embedding_model = SentenceTransformer(EMBED_MODEL, device=device)

if device == 'cuda':
    embedding_model = embedding_model.half()  # FP16: ~2x throughput, half VRAM

# ── Compute or load cached embeddings ────────────────────────────────────────
if os.path.exists(CACHE_PATH):
    print(f"Loading cached embeddings from '{CACHE_PATH}'...")
    embeddings = np.load(CACHE_PATH)
    print(f'✅ Loaded from cache — shape: {embeddings.shape}')
else:
    print('Computing embeddings...')
    batch_size = 256 if device == 'cuda' else (64 if device == 'mps' else 32)

    embeddings = embedding_model.encode(
        docs,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
        device=device,
    )

    np.save(CACHE_PATH, embeddings)
    print(f"✅ Done — shape: {embeddings.shape}. Cached to '{CACHE_PATH}'")

# ── Target variable ───────────────────────────────────────────────────────────
# Log-transform retweets to reduce skew (predict log1p, evaluate on original scale)
X = embeddings
y = df['retweets'].values

print(f'\nFeature matrix X : {X.shape}')
print(f'Target vector  y : {y.shape}')
print(f'Retweet stats    — min: {df["retweets"].min()}, max: {df["retweets"].max()}, mean: {df["retweets"].mean():.1f}')

▶ Using device: cpu
Computing embeddings...


Batches:   0%|          | 0/312 [00:00<?, ?it/s]

✅ Done — shape: (9976, 384). Cached to 'embeddings_cache.npy'

Feature matrix X : (9976, 384)
Target vector  y : (9976,)
Retweet stats    — min: 0, max: 100, mean: 49.7


## ✂️ 5. Train / Test Split

80% of data is used for training, 20% held out for evaluation.

In [7]:
# Create retweet range bins for stratification
y_bins, bin_edges = pd.qcut(y, q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'],
                             duplicates='drop', retbins=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y_bins,
)

print(f'✅ Train size : {X_train.shape[0]:>5} samples  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'   Test  size : {X_test.shape[0]:>5} samples  ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'   Feature dim: {X_train.shape[1]}')

# Verify stratification — distribution should look similar across splits
labels = [f'Q{i+1}' for i in range(len(bin_edges) - 1)]
train_bins = pd.cut(y_train, bins=bin_edges, labels=labels, include_lowest=True)
test_bins  = pd.cut(y_test,  bins=bin_edges, labels=labels, include_lowest=True)

print('\n   Retweet range distribution:')
print(f'   {"Quantile":<8} {"Range":<20} {"Train":>8} {"Test":>8}')
print(f'   {"-"*48}')
for i, label in enumerate(labels):
    range_str = f'{bin_edges[i]:.0f}–{bin_edges[i+1]:.0f}'
    train_pct = (train_bins == label).mean() * 100
    test_pct  = (test_bins  == label).mean() * 100
    print(f'   {label:<8} {range_str:<20} {train_pct:>7.1f}%  {test_pct:>7.1f}%')

✅ Train size :  7980 samples  (80.0%)
   Test  size :  1996 samples  (20.0%)
   Feature dim: 384

   Retweet range distribution:
   Quantile Range                   Train     Test
   ------------------------------------------------
   Q1       0–25                    25.8%     25.8%
   Q2       25–49                   24.3%     24.2%
   Q3       49–75                   25.9%     25.9%
   Q4       75–100                  24.1%     24.1%


## 🌲 6. Random Forest

A `RandomForestRegressor` is trained on the text embeddings to predict log-transformed retweet counts.

**Why Random Forest?**
- Robust to high-dimensional sparse feature spaces (384-dim embeddings)
- Built-in feature importance via impurity-based splits
- Naturally handles non-linear relationships between text semantics and engagement

Predictions are inverse-transformed (`expm1`) back to the original retweet scale for interpretable RMSE.

In [8]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---- Log transform target ----
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# ---- Random Forest Model ----
rf_model = RandomForestRegressor(
    max_depth=6,
    n_estimators=50,
    random_state=42
)

rf_model.fit(X_train, y_train_log)

# ---- Predictions ----
rf_y_train_pred_log = rf_model.predict(X_train)
rf_y_test_pred_log  = rf_model.predict(X_test)

# Convert back to original scale
rf_y_train_pred = np.expm1(rf_y_train_pred_log)
rf_y_test_pred  = np.expm1(rf_y_test_pred_log)

# Optional: clip negative predictions
rf_y_train_pred = np.clip(rf_y_train_pred, 0, None)
rf_y_test_pred  = np.clip(rf_y_test_pred, 0, None)

# ---- Metrics ----
rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf_y_train_pred))
rf_test_rmse  = np.sqrt(mean_squared_error(y_test,  rf_y_test_pred))
rf_train_mae  = mean_absolute_error(y_train, rf_y_train_pred)
rf_test_mae   = mean_absolute_error(y_test,  rf_y_test_pred)
rf_train_r2   = r2_score(y_train_log, rf_y_train_pred_log)  # log scale
rf_test_r2    = r2_score(y_test_log,  rf_y_test_pred_log)   # log scale


# ---- Print Results ----
print("🚀 Evaluation For Random Forest Regressor")
print("="*50)
print(f"{'Metric':<20}{'Train':>10}{'Test':>10}")
print("-"*50)
print(f"{'RMSE (original scale)':<20}{rf_train_rmse:>10.2f}{rf_test_rmse:>10.2f}")
print(f"{'MAE  (original scale)':<20}{rf_train_mae:>10.2f}{rf_test_mae:>10.2f}")
print(f"{'R² (log scale)':<20}{rf_train_r2:>10.4f}{rf_test_r2:>10.4f}")
print("="*50)


🚀 Evaluation For Random Forest Regressor
Metric                   Train      Test
--------------------------------------------------
RMSE (original scale)     31.10     31.41
MAE  (original scale)     26.23     26.35
R² (log scale)          0.0425   -0.0043


## 🚀 7. Gradient Boosting

A `GradientBoostingRegressor` is trained on the same text embeddings to predict log-transformed retweet counts.

**Why Gradient Boosting?**
- Sequentially corrects errors from prior trees — often outperforms Random Forest on tabular/embedding data
- Handles complex non-linear patterns through boosting
- Provides a diverse learner to complement Random Forest in the ensemble

Predictions are inverse-transformed (`expm1`) back to the original retweet scale.

In [9]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# ---- Log transform target ----
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

gb_model = GradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.05,
    max_depth=5,
    min_samples_leaf=5,
    subsample=0.8,
    random_state=42,
)
gb_model.fit(X_train, y_train_log)

# ---- Predictions ----
gb_y_train_pred_log = gb_model.predict(X_train)
gb_y_test_pred_log  = gb_model.predict(X_test)

# Convert back to original scale
gb_y_train_pred = np.expm1(gb_y_train_pred_log)
gb_y_test_pred  = np.expm1(gb_y_test_pred_log)

# Optional: clip negative predictions
gb_y_train_pred = np.clip(gb_y_train_pred, 0, None)
gb_y_test_pred  = np.clip(gb_y_test_pred, 0, None)

# ---- Metrics ----
gb_train_rmse = np.sqrt(mean_squared_error(y_train, gb_y_train_pred))
gb_test_rmse  = np.sqrt(mean_squared_error(y_test,  gb_y_test_pred))
gb_train_mae  = mean_absolute_error(y_train, gb_y_train_pred)
gb_test_mae   = mean_absolute_error(y_test,  gb_y_test_pred)
gb_train_r2   = r2_score(y_train_log, gb_y_train_pred_log)  # log scale
gb_test_r2    = r2_score(y_test_log,  gb_y_test_pred_log)   # log scale

# ---- Print Results ----
print("🚀 Evaluation For Gradient Boosting Regressor")
print("="*50)
print(f"{'Metric':<20}{'Train':>10}{'Test':>10}")
print("-"*50)
print(f"{'RMSE (original scale)':<20}{gb_train_rmse:>10.2f}{gb_test_rmse:>10.2f}")
print(f"{'MAE  (original scale)':<20}{gb_train_mae:>10.2f}{gb_test_mae:>10.2f}")
print(f"{'R² (log scale)':<20}{gb_train_r2:>10.4f}{gb_test_r2:>10.4f}")
print("="*50)

🚀 Evaluation For Gradient Boosting Regressor
Metric                   Train      Test
--------------------------------------------------
RMSE (original scale)     30.24     31.59
MAE  (original scale)     25.33     26.46
R² (log scale)          0.1242   -0.0099


## 🗳️ 9. Ensemble (Soft Voting Average)

The final prediction is the **arithmetic average** of the Random Forest and Gradient Boosting predictions (soft voting).

**Why ensemble averaging?**
- Reduces variance: averaged predictions are less sensitive to individual model noise
- The two models have different inductive biases (bagging vs. boosting), so their errors are partially uncorrelated
- Averaging in log-space before inverse-transforming preserves the geometric mean — appropriate for skewed count targets

A side-by-side comparison table summarises all three models.

In [10]:
# ── Soft voting: average predictions in log-space, then inverse-transform ─────
ensemble_pred_train_log = (rf_y_train_pred_log + gb_y_train_pred_log) / 2
ensemble_pred_test_log  = (rf_y_test_pred_log  + gb_y_test_pred_log)  / 2

ensemble_y_pred_train = np.expm1(ensemble_pred_train_log)
ensemble_y_pred_test  = np.expm1(ensemble_pred_test_log)

# ── Metrics ───────────────────────────────────────────────────────────────────
ens_train_rmse = np.sqrt(mean_squared_error(y_train, ensemble_y_pred_train))
ens_test_rmse  = np.sqrt(mean_squared_error(y_test,  ensemble_y_pred_test))
ens_train_mae  = mean_absolute_error(y_train, ensemble_y_pred_train)
ens_test_mae   = mean_absolute_error(y_test,  ensemble_y_pred_test)
ens_train_r2   = r2_score(y_train_log, ensemble_pred_train_log)
ens_test_r2    = r2_score(y_test_log,  ensemble_pred_test_log)

print()
print('=' * 70)
print('🗳️  ENSEMBLE (SOFT VOTING AVERAGE RF + GBM) RESULTS')
print('=' * 70)
print(f"{'Metric':<20}{'Train':>10}{'Test':>10}")
print(f"{'-'*44}")
print(f"{'RMSE (retweets)':<20}{ens_train_rmse:>10.2f}{ens_test_rmse:>10.2f}")
print(f"{'MAE  (retweets)':<20}{ens_train_mae:>10.2f}{ens_test_mae:>10.2f}")
print(f"{'R²   (log scale)':<20}{ens_train_r2:>10.4f}{ens_test_r2:>10.4f}")
print('=' * 70)

# ── Side-by-side summary ──────────────────────────────────────────────────────
print()
print('=' * 70)
print('📊 MODEL COMPARISON SUMMARY (Test Set)')
print('=' * 70)

summary = pd.DataFrame({
    'Model':            ['Random Forest', 'Gradient Boosting', 'Ensemble (avg RF+GBM)'],
    'Train RMSE':       [rf_train_rmse,   gb_train_rmse,       ens_train_rmse],
    'Test  RMSE':       [rf_test_rmse,    gb_test_rmse,        ens_test_rmse],
    'Train MAE':        [rf_train_mae,    gb_train_mae,        ens_train_mae],
    'Test  MAE':        [rf_test_mae,     gb_test_mae,         ens_test_mae],
    'Train R² (log)':   [rf_train_r2,     gb_train_r2,         ens_train_r2],
    'Test  R² (log)':   [rf_test_r2,      gb_test_r2,          ens_test_r2],
})

# Round metrics for display
for col in summary.columns[1:]:
    summary[col] = summary[col].round(4)

# Print nicely
from tabulate import tabulate
print(tabulate(summary, headers='keys', tablefmt='github', showindex=False, floatfmt='.4f'))

# ── Highlight best test RMSE ──────────────────────────────────────────────────
best_idx  = summary['Test  RMSE'].idxmin()
best_name = summary.loc[best_idx, 'Model']
best_rmse = summary.loc[best_idx, 'Test  RMSE']
print(f'\n🏆 Best test RMSE: {best_name}  ({best_rmse:.4f})')


🗳️  ENSEMBLE (SOFT VOTING AVERAGE RF + GBM) RESULTS
Metric                   Train      Test
--------------------------------------------
RMSE (retweets)          30.64     31.49
MAE  (retweets)          25.76     26.39
R²   (log scale)        0.0853   -0.0053

📊 MODEL COMPARISON SUMMARY (Test Set)
| Model                 |   Train RMSE |   Test  RMSE |   Train MAE |   Test  MAE |   Train R² (log) |   Test  R² (log) |
|-----------------------|--------------|--------------|-------------|-------------|------------------|------------------|
| Random Forest         |      31.1035 |      31.4134 |     26.2321 |     26.3532 |           0.0425 |          -0.0043 |
| Gradient Boosting     |      30.2389 |      31.5874 |     25.3279 |     26.4561 |           0.1242 |          -0.0099 |
| Ensemble (avg RF+GBM) |      30.6370 |      31.4855 |     25.7576 |     26.3920 |           0.0853 |          -0.0053 |

🏆 Best test RMSE: Random Forest  (31.4134)


## 🔁 10. Prediction Pipelines

Two end-to-end pipelines — one per base model — take a raw tweet string and return:

1. **Predicted retweet count** (original scale, via `expm1`)  

- **Clean** — `clean_text()` removes URLs, mentions, hashtag symbols, RT prefix, and special characters  
- **Embed** — `embedding_model.encode()` produces a 384-dimensional L2-normalized sentence vector 

Each pipeline applies the same steps in order:
- **Predict** — the fitted regressor predicts in **log-space**, and `expm1` converts it back to retweet scale  

A **10-tweet demo** (any sample tweets) is run through both pipelines, with a **side-by-side comparison table** showing predicted retweet counts for RF, GBM, and their Ensemble average.

In [11]:
import numpy as np
from sklearn.pipeline import Pipeline

# ── 1. Build the Pipeline ───────────────────────────────────────────────────
# Ensure commas are present between every tuple in the list
rf_pipeline_model = Pipeline([
    ('rf_model', rf_model)  # Ensure rf_model is already defined
])

# ── 4. Build the GBM Pipeline similarly ──────────────────────────────────────
gb_pipeline_model = Pipeline([
    ('gb_model', gb_model)  # Ensure gb_model is already defined
])

# ── Fit Pipelines on training tweets ─────────────────────────────────────────
rf_pipeline_model.fit(X_train, np.log1p(y_train))
gb_pipeline_model.fit(X_train, np.log1p(y_train))


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('gb_model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",50
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",0.8
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for f

In [12]:
import pickle
import os

# ── 1. Define File Names ─────────────────────────────────────────────────────
rf_filename      = '../models/rf_pipeline_model.pkl'
gb_filename      = '../models/gb_pipeline_model.pkl'
embedder_filename = '../models/embedder.pkl'

# ── 2. Save the Random Forest Pipeline ───────────────────────────────────────
with open(rf_filename, 'wb') as f:
    pickle.dump(rf_pipeline_model, f)
print(f"✅ Saved Random Forest Pipeline to: {rf_filename}")

# ── 3. Save the Gradient Boosting Pipeline ──────────────────────────────────
with open(gb_filename, 'wb') as f:
    pickle.dump(gb_pipeline_model, f)
print(f"✅ Saved GBM Pipeline to: {gb_filename}")

# ── 4. Save the Embedding Model ──────────────────────────────────────────────
with open(embedder_filename, 'wb') as f:
    pickle.dump(embedding_model, f)
print(f"✅ Saved Embedding Model to: {embedder_filename}")

# ── 5. Verify File Sizes ─────────────────────────────────────────────────────
rf_size       = os.path.getsize(rf_filename)       / (1024 * 1024)
gb_size       = os.path.getsize(gb_filename)       / (1024 * 1024)
embedder_size = os.path.getsize(embedder_filename) / (1024 * 1024)
print(f"📊 RF Model Size      : {rf_size:.2f} MB")
print(f"📊 GBM Model Size     : {gb_size:.2f} MB")
print(f"📊 Embedder Model Size: {embedder_size:.2f} MB")

✅ Saved Random Forest Pipeline to: ../models/rf_pipeline_model.pkl
✅ Saved GBM Pipeline to: ../models/gb_pipeline_model.pkl
✅ Saved Embedding Model to: ../models/embedder.pkl
📊 RF Model Size      : 0.16 MB
📊 GBM Model Size     : 0.12 MB
📊 Embedder Model Size: 66.83 MB


In [13]:

# ── Sample 10 Tweets (5 viral, 5 non-viral) ─────────────────────────────────
sample_tweets = [
    # ── Likely Viral ─────────────────────────────────────────────────────────
    "🚨 BREAKING: Scientists just confirmed a major breakthrough in Alzheimer's treatment. "
    "First trial shows 94% reduction in plaques. This is the moment we've been waiting for. RT to spread hope. 🧠",

    "I gave ChatGPT my entire 10-year career plan and asked it to destroy it. "
    "What it said back changed how I think about everything. Thread 🧵👇",

    "Unpopular opinion: The 9-to-5 was never designed for human beings to thrive. "
    "It was designed for factories. We kept the hours. We lost the purpose. Change my mind.",

    "This 12-year-old built a water filtration system from trash for her village in Kenya. "
    "She was rejected from 3 science fairs before winning a UN award. "
    "Her name is Amara. Remember it. 🌍❤️",

    "Netflix just quietly removed the ability to download content offline. "
    "No announcement. No email. Just gone. This is exactly why people pirate. 🏴‍☠️",

    # ── Unlikely to Go Viral ─────────────────────────────────────────────────
    "Just updated my laptop drivers. Everything seems to be working fine now.",

    "Had oatmeal for breakfast again this morning. Added some blueberries this time.",

    "Reminder to myself: pick up dry cleaning before Thursday.",

    "The quarterly synergy report has been uploaded to the shared drive. "
    "Please review section 4b before Friday's alignment meeting.",

    "Light rain expected in the afternoon. Moderate temperatures. "
    "Good day to stay indoors and get some work done.",
]

# ── Run Pipelines ─────────────────────────────────────────────────────────────
rows = []
for tweet in sample_tweets:
    # RF prediction
    tweet = clean_text(tweet)  # Ensure same cleaning as during training
    embedding = embedding_model.encode([tweet], normalize_embeddings=True, convert_to_numpy=True)[0].reshape(1, -1)
    rf_log = rf_pipeline_model.predict(embedding)[0]
    rf_pred = max(0, round(np.expm1(rf_log)))
    
    # GBM prediction
    gb_log = gb_pipeline_model.predict(embedding)[0]
    gb_pred = max(0, round(np.expm1(gb_log)))
    
    # Ensemble: average log-space
    ens_pred = max(0, round(np.expm1((rf_log + gb_log)/2)))
    
    rows.append({
        "Tweet"       : tweet[:65] + ("…" if len(tweet) > 65 else ""),
        "RF RT"       : rf_pred,
        "GBM RT"      : gb_pred,
        "Ensemble RT" : ens_pred
    })

# ── Display Results ──────────────────────────────────────────────────────────
print("="*120)
print("🐦  10-TWEET PREDICTIONS — RF | GBM | ENSEMBLE (Original Scale)")
print("="*120)
print(tabulate(rows, headers="keys", tablefmt="github"))

🐦  10-TWEET PREDICTIONS — RF | GBM | ENSEMBLE (Original Scale)
| Tweet                                                              |   RF RT |   GBM RT |   Ensemble RT |
|--------------------------------------------------------------------|---------|----------|---------------|
| breaking scientists just confirmed a major breakthrough in alzhei… |      35 |       32 |            33 |
| i gave chatgpt my entire 10year career plan and asked it to destr… |      37 |       38 |            37 |
| unpopular opinion the 9to5 was never designed for human beings to… |      37 |       33 |            35 |
| this 12yearold built a water filtration system from trash for her… |      36 |       34 |            35 |
| netflix just quietly removed the ability to download content offl… |      36 |       36 |            36 |
| just updated my laptop drivers everything seems to be working fin… |      33 |       34 |            33 |
| had oatmeal for breakfast again this morning added some blueberri… |   

## 💾 Metadata Export

Save a JSON file containing the information needed to **monitor the deployed model for data drift** and to **reproduce the virality classifier** in any downstream service:

| Field | Purpose |
|---|---|
| `embedding_mean` | Per-dimension mean of all training embeddings — baseline centroid for drift detection |
| `embedding_std` | Per-dimension std of all training embeddings — expected spread; large deviations signal drift |
| `embedding_dim` | Sanity-check: expected vector length (384 for MiniLM-L12) |
| `virality_threshold` | Retweet count used to separate viral from non-viral — must travel with any deployed pipeline |
| `virality_percentile` | Documents how the threshold was derived (75th-percentile of training retweets) |
| `embed_model` | Model name so the same tokeniser/encoder can be loaded at inference time |
| `trained_on_n_samples` | Training set size — useful for drift severity calibration |
| `retweet_stats` | Training-set retweet distribution (min/max/mean/median/p75/p95) for reference |

The file is written to `model_metadata.json` in the current working directory.


In [14]:
import json, os
import numpy as np

# ── 1. Embedding statistics ───────────────────────────────────────────────────
embed_mean = X_train.mean(axis=0)          # shape (384,)
embed_std  = X_train.std(axis=0)           # shape (384,)

# ── 2. Training-set retweet distribution ─────────────────────────────────────
rt_orig = np.expm1(y_train)

retweet_stats = {
    "min"    : int(rt_orig.min()),
    "max"    : int(rt_orig.max()),
    "mean"   : int(rt_orig.mean()),
    "median" : int(np.median(rt_orig)),
    "p75"    : int(np.percentile(rt_orig, 75)),
    "p95"    : int(np.percentile(rt_orig, 95)),
}

# ── 3. Feature statistics (drift detection) ──────────────────────────────────
feature_stats = {
    "embedding_mean" : embed_mean.tolist(),
    "embedding_std"  : embed_std.tolist(),
    "embedding_dim"  : int(embed_mean.shape[0]),
}

# ── 4. Model metadata (provenance + API info) ─────────────────────────────────
model_metadata = {
    # Virality classifier
    "virality_threshold"    : retweet_stats["p75"],
    "virality_percentile"   : int(75),

    # Model provenance
    "embed_model"           : EMBED_MODEL,
    "trained_on_n_samples"  : int(X_train.shape[0]),

    # Retweet distribution reference
    "retweet_stats"         : retweet_stats,

    # API /model-info fields
    "input"                 : "raw tweet text (embedded internally)",
    "pipeline_steps"        : ["Text Cleaning", "Embedding", "Regression"],
    "training_samples"      : int(X_train.shape[0]),
    "test_samples"          : int(X_test.shape[0]),
}

# ── 5. Write to disk ──────────────────────────────────────────────────────────
FEATURE_STATS_PATH  = "../models/feature_stats.json"
MODEL_METADATA_PATH = "../models/model_metadata.json"

with open(FEATURE_STATS_PATH, "w") as f:
    json.dump(feature_stats, f, indent=2)
print(f"✅ Feature statistics saved to '{os.path.abspath(FEATURE_STATS_PATH)}'")

with open(MODEL_METADATA_PATH, "w") as f:
    json.dump(model_metadata, f, indent=2)
print(f"✅ Model metadata saved to '{os.path.abspath(MODEL_METADATA_PATH)}'")

print()
print("── Summary ─────────────────────────────────────────────────────────────")
print(f"  Embedding dim           : {feature_stats['embedding_dim']}")
print(f"  Embedding mean (first 5): {[round(v, 6) for v in feature_stats['embedding_mean'][:5]]}")
print(f"  Embedding std  (first 5): {[round(v, 6) for v in feature_stats['embedding_std'][:5]]}")
print(f"  Virality threshold      : {model_metadata['virality_threshold']:,.1f} retweets "
      f"({model_metadata['virality_percentile']}th-percentile)")
print(f"  Trained on              : {model_metadata['trained_on_n_samples']:,} samples")
print(f"  Test samples            : {model_metadata['test_samples']:,} samples")
print(f"  Retweet stats (train)   : {retweet_stats}")
print()
print("── Drift-detection usage guide ─────────────────────────────────────────")
print("  At inference, embed the new tweet and compute:")
print("    z = (new_embedding - embedding_mean) / embedding_std")
print("  If mean(|z|) > 3  →  the tweet may be out-of-distribution (drift alert).")

✅ Feature statistics saved to '/home/kunalpop/Desktop/tweet-analytics/models/feature_stats.json'
✅ Model metadata saved to '/home/kunalpop/Desktop/tweet-analytics/models/model_metadata.json'

── Summary ─────────────────────────────────────────────────────────────
  Embedding dim           : 384
  Embedding mean (first 5): [-0.015863, 0.022651, 0.012494, -0.009584, -0.002121]
  Embedding std  (first 5): [0.043178, 0.043883, 0.04426, 0.047869, 0.048139]
  Virality threshold      : 373,324,199,679,900,148,487,366,050,316,288.0 retweets (75th-percentile)
  Trained on              : 7,980 samples
  Test samples            : 1,996 samples
  Retweet stats (train)   : {'min': 0, 'max': 26881171418161356094253400435962903554686976, 'mean': 402867935356087966952709101058064279142400, 'median': 1907346572495099527168, 'p75': 373324199679900148487366050316288, 'p95': 181123908288902334832971246827918212464640}

── Drift-detection usage guide ─────────────────────────────────────────
  At inferenc